In [22]:
# scores = []
# for i in range(len(data)):
#     scores.append({data[i]['date'] : minute_analysis(data[i]['full_text'])})
# with open("scores.json", "w") as f:
#     json.dump(scores, f, indent=2)


# macro_df = pd.DataFrame()
# for i in range(len(data)):
#     macro_df[i] = fetch_macro_indicators(data[i]['date'])

# macro_df = macro_df.T 

# macro_df.to_json('macro_indicators.json', orient='records', indent=2)
from data_gathering import fetch_macro_indicators, parse_first_day
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

import json
# Load your data
file_path = 'fomc_training_data.json'
with open(file_path, 'r') as f:
    data = json.load(f)

n_meetings = len(data) - 1 

with open('scores.json', 'r') as f:
    minutes_data = json.load(f)

for item in minutes_data:
    for old_key in list(item.keys()):
        new_key = parse_first_day(old_key)
        if new_key != old_key:
            item[new_key] = item.pop(old_key)

minutes_list = []
for item in minutes_data[:-1]:  # Take first N-1 meetings
    date_label = list(item.keys())[0]
    features = item[date_label]
    features['date_label'] = date_label 
    minutes_list.append(features)

minutes_df = pd.DataFrame(minutes_list)
minutes_df['date'] = pd.to_datetime(minutes_df['date_label']).dt.date
minutes_df = minutes_df.drop(columns=['date_label'])

# Get macro data for meetings 1 to N-1 (current conditions when decisions made)
macro_df = pd.read_json('macro_indicators.json', orient='records')
macro_df = macro_df.iloc[1:n_meetings+1].reset_index(drop=True)  # Take rows 1 to N-1

# Labels: decisions from meetings 1 to N-1 
label_map = {"cut": 0, "hold": 1, "hike": 2}
y = np.array([label_map[data[i]["decision"]] for i in range(1, n_meetings+1)], dtype=np.int32)

# Combine features
combined_df = minutes_df.join(macro_df, lsuffix='_minutes', rsuffix='_macro')
combined_df = combined_df.drop(['date'], axis=1, errors='ignore')

# Now everything should have the same length
print(f"Minutes shape: {minutes_df.shape}")
print(f"Macro shape: {macro_df.shape}")  
print(f"Combined shape: {combined_df.shape}")
print(f"Labels length: {len(y)}")

# Convert to features
Phi = np.asarray(combined_df, dtype=np.float32)

# Verify shapes match
print(f"Phi shape: {Phi.shape}, y shape: {y.shape}")
assert Phi.shape[0] == len(y), f"Mismatch: {Phi.shape[0]} vs {len(y)}"

print("Checking for NaN values:")
print(f"Phi has NaN: {np.isnan(Phi).any()}")
print(f"Phi has inf: {np.isinf(Phi).any()}")
print(f"Phi min/max: {Phi.min():.4f} / {Phi.max():.4f}")


Minutes shape: (55, 12)
Macro shape: (55, 6)
Combined shape: (55, 17)
Labels length: 55
Phi shape: (55, 17), y shape: (55,)
Checking for NaN values:
Phi has NaN: False
Phi has inf: False
Phi min/max: -50.0000 / 29825.1816


In [23]:

combined_df

,policy_bias,guidance_strength,expected_move_bps,p_cut,p_hold,p_hike,inflation_tone,labor_tone,growth_tone,financial_conditions_tone,balance_sheet_signal,unemployement,core_cpi,gdp,ten_two_spread,fed_funds,vix
0,0.00,0.00,0,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.50,4.0,256.271,20328.553,0.55,1.51,18.20
1,0.80,0.60,25,0.00,0.20,0.80,0.70,0.80,0.80,0.70,0.70,3.8,257.145,20580.912,0.47,1.70,15.49
2,0.75,0.80,25,0.00,0.10,0.90,0.70,0.85,0.70,0.60,0.80,4.0,257.399,20580.912,0.42,1.82,12.34
3,0.80,0.10,25,0.05,0.15,0.80,0.70,0.90,0.80,0.60,0.70,3.8,257.699,20798.730,0.29,1.91,12.83
4,0.75,0.40,25,0.05,0.15,0.80,0.10,0.20,0.80,0.60,0.50,3.7,258.368,20798.730,0.27,1.95,12.42
5,0.80,0.90,25,0.00,0.05,0.95,0.60,0.90,0.85,0.60,0.80,3.8,259.439,20917.867,0.26,2.20,16.36
6,0.10,0.10,0,0.00,1.00,0.00,0.00,0.00,0.40,0.70,0.50,3.9,260.063,20917.867,0.17,2.27,25.58
7,0.30,0.40,25,0.00,0.10,0.90,0.00,0.00,0.20,0.80,0.70,4.0,260.766,21111.600,0.16,2.40,19.13
8,0.00,0.00,0,0.00,1.00,0.00,0.50,0.50,0.50,0.50,0.50,3.8,261.567,21111.600,0.15,2.41,13.56
9,0.00,0.90,0,0.00,1.00,0.00,0.00,0.00,0.00,0.40,-0.60,3.7,261.997,21397.938,0.24,2.42,13.12


In [21]:
macro_df = pd.read_json('macro_indicators.json', orient='records')
print(macro_df.head())
macro_df = macro_df.iloc[1:n_meetings+1].reset_index(drop=True)
macro_df.head()

   unemployement  core_cpi        gdp  ten_two_spread  fed_funds    vix
0            4.0   255.204  20328.553            0.60       1.41  14.79
1            4.0   256.271  20328.553            0.55       1.51  18.20
2            3.8   257.145  20580.912            0.47       1.70  15.49
3            4.0   257.399  20580.912            0.42       1.82  12.34
4            3.8   257.699  20798.730            0.29       1.91  12.83


,unemployement,core_cpi,gdp,ten_two_spread,fed_funds,vix
0,4.0,256.271,20328.553,0.55,1.51,18.20
1,3.8,257.145,20580.912,0.47,1.70,15.49
2,4.0,257.399,20580.912,0.42,1.82,12.34
3,3.8,257.699,20798.730,0.29,1.91,12.83
4,3.7,258.368,20798.730,0.27,1.95,12.42


In [18]:
fetch_macro_indicators('2018/3/20')

ValueError: Unrecognized date format: 2018/3/20

In [ ]:
# 2. Check class distribution
print("Label distribution:")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  {label}: {count}")

# 3. Check training history
print("Training history:")

scaler = MinMaxScaler()
Phi = scaler.fit_transform(Phi).astype(np.float32)

Phi_train, Phi_test, y_train, y_test = train_test_split(
    Phi, y, test_size=0.3, random_state=42,
    stratify=y if len(np.unique(y)) > 1 else None)

In [3]:
import pandas as pd

df = pd.read_csv('/Users/dylantoplas/Downloads/2025-09-MD.csv')
df

,sasdate,RPI,W875RX1,DPCERA3M086SBEA,CMRMTSPLx,RETAILx,INDPRO,IPFPNSS,IPFINAL,IPCONGD,...,DNDGRG3M086SBEA,DSERRG3M086SBEA,CES0600000008,CES2000000008,CES3000000008,UMCSENTx,DTCOLNVHFNM,DTCTHFNM,INVEST,VIXCLSx
0,Transform:,5.000,5.0,5.000,5.000000e+00,5.00000,5.0000,5.0000,5.0000,5.0000,...,6.000,6.000,6.00,6.00,6.00,2.0,6.00,6.00,6.0000,1.0000
1,1/1/1959,2583.560,2426.0,15.188,2.766768e+05,17689.23968,21.9616,23.3868,22.2620,31.6664,...,18.294,10.152,2.13,2.45,2.04,NaN,6476.00,12298.00,84.2043,NaN
2,2/1/1959,2593.596,2434.8,15.346,2.787140e+05,17819.01912,22.3917,23.7024,22.4549,31.8987,...,18.302,10.167,2.14,2.46,2.05,NaN,6476.00,12298.00,83.5280,NaN
3,3/1/1959,2610.396,2452.7,15.491,2.777753e+05,17967.91336,22.7142,23.8459,22.5651,31.8987,...,18.289,10.185,2.15,2.45,2.07,NaN,6508.00,12349.00,81.6405,NaN
4,4/1/1959,2627.446,2470.0,15.435,2.833627e+05,17978.97983,23.1981,24.1903,22.8957,32.4019,...,18.300,10.221,2.16,2.47,2.08,NaN,6620.00,12484.00,81.8099,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
796,4/1/2025,20698.762,16739.9,123.748,1.555663e+06,721789.00000,103.6224,101.3671,101.1112,101.6979,...,119.658,131.767,32.22,36.96,28.78,52.2,554180.13,940362.47,5500.0706,32.5116
797,5/1/2025,20581.743,16703.7,123.575,1.550522e+06,716101.00000,103.6570,101.4038,101.1856,101.5808,...,119.780,132.071,32.31,37.08,28.87,52.2,551829.26,938763.49,5526.3170,20.3620
798,6/1/2025,20575.971,16664.7,123.894,1.556845e+06,723033.00000,104.2115,101.7271,101.5445,101.9628,...,120.208,132.386,32.40,37.23,28.94,60.7,549682.41,937344.92,5555.2136,18.3246
799,7/1/2025,20625.729,16718.9,124.370,1.565742e+06,727414.00000,103.8194,101.4573,101.4961,101.7345,...,120.036,132.778,32.47,37.28,29.01,61.7,547389.12,934567.19,5585.9624,16.4718


In [6]:
features = [
    'FEDFUNDS',      # Effective Federal Funds Rate: 
    'CPIAUCSL',      # Consumer Price Index: average change in prices over time for a "market basket"
    'INDPRO',        # Industrial Production
    'UNRATE',        # Unemployment Rate
    'M2SL',          # M2 Money Supply
    'S&P 500'        # Stock returns proxy
]

X = df[features]
y = df['GS10']